# 🤖 Models & Prompts — Hands-On Examples

> **Module 03 | Submodule 02 | Practical Notebook**
>
> This notebook covers:
> - LLM vs ChatModel interface comparison
> - Working with multiple model providers
> - PromptTemplate and ChatPromptTemplate
> - Multi-turn conversations with MessagesPlaceholder
> - Few-shot prompting
> - Provider benchmarking on the same task

In [ ]:
!pip install langchain langchain-openai langchain-anthropic \
             langchain-google-genai langchain-ollama \
             langchain-groq python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "module-03-models-prompts"
print("Ready!")

---
## 1️⃣ LLM vs ChatModel

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Input Method 1: string (simplest)
r1 = llm.invoke("What is LangChain in one sentence?")
print("String input:", r1.content)
print("Output type:", type(r1).__name__)

In [ ]:
# Input Method 2: typed message objects
messages = [
    SystemMessage(content="You are a concise technical writer."),
    HumanMessage(content="What is LangChain in one sentence?")
]
r2 = llm.invoke(messages)
print("With system prompt:", r2.content)

# Input Method 3: tuple shorthand
r3 = llm.invoke([
    ("system", "You are a concise technical writer."),
    ("human",  "What is LangChain in one sentence?")
])
print("Tuple format:", r3.content)

In [ ]:
# Explore the AIMessage output
response = llm.invoke("What is LangGraph?")

print("Content:",          response.content[:80])
print("Type:",             response.type)
print("Token usage:",      response.usage_metadata)
print("Model metadata:",   response.response_metadata.get('model_name'))
print("Finish reason:",    response.response_metadata.get('finish_reason'))
print("Tool calls:",       response.tool_calls)  # [] when no tools

In [ ]:
# Streaming — token by token
print("Streaming output:")
for chunk in llm.stream("Explain LangChain in 3 bullet points:"):
    print(chunk.content, end="", flush=True)
print()  # newline

In [ ]:
# Batch — parallel processing
import time

questions = [
    "What is LangChain?",
    "What is LangGraph?",
    "What is LangSmith?",
]

start = time.time()
results = llm.batch(questions)
elapsed = time.time() - start

print(f"Processed {len(results)} questions in {elapsed:.2f}s (parallel)")
for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r.content[:60]}...")
    print()

---
## 2️⃣ Multiple Model Providers

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Set up multiple providers
providers = {
    "OpenAI GPT-4o-mini": ChatOpenAI(model="gpt-4o-mini", temperature=0),
}

# Add Anthropic if API key available
if os.getenv("ANTHROPIC_API_KEY"):
    from langchain_anthropic import ChatAnthropic
    providers["Anthropic Claude Haiku"] = ChatAnthropic(
        model="claude-3-haiku-20240307", temperature=0
    )

# Add Google if API key available
if os.getenv("GOOGLE_API_KEY"):
    from langchain_google_genai import ChatGoogleGenerativeAI
    providers["Google Gemini Flash"] = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash", temperature=0
    )

# Build the SAME chain for all providers
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human",  "{question}")
])
parser = StrOutputParser()

question = {"question": "What makes a good AI agent?"}

# Compare all providers — SAME chain code!
print("=" * 60)
for name, llm_provider in providers.items():
    chain = prompt | llm_provider | parser
    result = chain.invoke(question)
    print(f"\n🤖 {name}:")
    print(result[:200])
    print("-" * 60)

---
## 3️⃣ PromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate

# Basic PromptTemplate
template = PromptTemplate.from_template(
    "Write a {tone} {length} about {topic}."
)

print("Variables:", template.input_variables)

# Format it
formatted = template.format(tone="technical", length="overview", topic="LangChain")
print("Formatted:", formatted)

In [ ]:
# Partial variables — pre-fill some now
template = PromptTemplate.from_template(
    "Translate '{text}' from {source_lang} to {target_lang}."
)

# Fix the languages — reuse this template for many translations
en_to_fr = template.partial(source_lang="English", target_lang="French")
en_to_de = template.partial(source_lang="English", target_lang="German")

# Only need to provide {text} later
print(en_to_fr.format(text="Hello, how are you?"))
print(en_to_de.format(text="Hello, how are you?"))

In [ ]:
# PromptTemplate in a chain
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

translate_chain = en_to_fr | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()
result = translate_chain.invoke({"text": "LangChain makes building AI applications easy."})
print("French:", result)

---
## 4️⃣ ChatPromptTemplate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Basic ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} specializing in {domain}."),
    ("human",  "{question}")
])

print("Input variables:", prompt.input_variables)

# Format to messages
messages = prompt.format_messages(
    role="data scientist",
    domain="machine learning",
    question="What is overfitting?"
)
for msg in messages:
    print(f"[{msg.type}]: {msg.content}")

In [ ]:
# Build an expert advisor with different specializations
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

expert_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a {specialization} expert.
Your communication style: {style}
Keep answers to 2-3 sentences maximum."""),
    ("human", "{question}")
])

llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = expert_prompt | llm | StrOutputParser()

# Same question, different experts
configs = [
    {"specialization": "Python developer",    "style": "technical and precise"},
    {"specialization": "high school teacher", "style": "simple and relatable"},
    {"specialization": "business executive",  "style": "outcomes-focused"},
]

question = "What is machine learning?"
for config in configs:
    config["question"] = question
    result = chain.invoke(config)
    print(f"\n🎭 As {config['specialization']}:")
    print(result)

---
## 5️⃣ Multi-Turn Conversations with MessagesPlaceholder

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Prompt with conversation history slot
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Remember everything the user tells you."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = chat_prompt | llm | StrOutputParser()

# Simulate a multi-turn conversation
history = []
conversations = [
    "Hi! My name is Bob and I'm a machine learning engineer.",
    "I work at a startup building RAG systems.",
    "What's my name and what do I do?",  # Tests memory
]

for user_msg in conversations:
    response = chain.invoke({"history": history, "question": user_msg})
    print(f"User: {user_msg}")
    print(f"AI:   {response}")
    print()

    # Update history
    history.append(HumanMessage(content=user_msg))
    history.append(AIMessage(content=response))

---
## 6️⃣ Few-Shot Prompting

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Define examples for the LLM to learn from
examples = [
    {"input": "langchain",  "output": "LangChain"},
    {"input": "langgraph",  "output": "LangGraph"},
    {"input": "langsmith",  "output": "LangSmith"},
    {"input": "openai",     "output": "OpenAI"},
    {"input": "chatgpt",    "output": "ChatGPT"},
]

# Format for each example
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Capitalize this AI/tech term correctly: {input}"),
    ("ai",    "{output}")
])

# Few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Full prompt with few-shot examples
full_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a precise formatter for AI and tech term names. Output ONLY the correctly capitalized term."),
    few_shot_prompt,
    ("human", "Capitalize this AI/tech term correctly: {input}")
])

chain = full_prompt | llm | StrOutputParser()

# Test with new terms
test_terms = ["anthropic", "huggingface", "langflow", "gpt4", "pytorch"]
for term in test_terms:
    result = chain.invoke({"input": term})
    print(f"{term:15} → {result}")

In [ ]:
# Few-shot for sentiment classification
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", """Classify reviews as: positive, negative, or neutral.
Only output one word.

Examples:
Review: "Absolutely loved it!" → positive
Review: "Complete waste of money." → negative
Review: "It arrived on time." → neutral
Review: "Broke after two days." → negative
Review: "Works as expected, nothing more." → neutral"""),
    ("human", "Review: \"{review}\" →")
])

sentiment_chain = sentiment_prompt | llm | StrOutputParser()

reviews = [
    "This is the best product I've ever bought!",
    "It stopped working after a week.",
    "Decent product for the price.",
    "Absolutely terrible customer service.",
    "Does what it says on the box.",
]

for review in reviews:
    sentiment = sentiment_chain.invoke({"review": review})
    emoji = {"positive": "✅", "negative": "❌", "neutral": "⚪"}.get(sentiment.strip(), "❓")
    print(f"{emoji} {sentiment.strip():10} | {review[:50]}")

---
## 7️⃣ Chain-of-Thought Prompting

In [ ]:
# Without CoT
simple_prompt = ChatPromptTemplate.from_messages([
    ("human", "Answer directly: {question}")
])

# With Chain-of-Thought
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """Think through problems step by step.
Format:
Step 1: [understand the problem]
Step 2: [work through it]
Step 3: [verify]
Answer: [final answer]"""),
    ("human", "{question}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

problem = """A LangChain application processes 150 requests per hour.
Each request costs $0.002 in API fees.
The app runs 8 hours a day, 5 days a week.
What is the weekly API cost?"""

print("=== WITHOUT CoT ===")
chain = simple_prompt | llm | StrOutputParser()
print(chain.invoke({"question": problem}))

print("\n=== WITH CoT ===")
chain = cot_prompt | llm | StrOutputParser()
print(chain.invoke({"question": problem}))

---
## 8️⃣ Provider Benchmarking — Same Task, Multiple LLMs

In [ ]:
import time

benchmark_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Python expert. Write clean, efficient code."),
    ("human",  "{task}")
])

parser = StrOutputParser()
task = {"task": "Write a Python function that finds all prime numbers up to n using the Sieve of Eratosthenes."}

# Benchmark each available provider
providers_to_test = {
    "GPT-4o-mini": ChatOpenAI(model="gpt-4o-mini", temperature=0),
}

if os.getenv("ANTHROPIC_API_KEY"):
    from langchain_anthropic import ChatAnthropic
    providers_to_test["Claude Haiku"] = ChatAnthropic(model="claude-3-haiku-20240307", temperature=0)

if os.getenv("GOOGLE_API_KEY"):
    from langchain_google_genai import ChatGoogleGenerativeAI
    providers_to_test["Gemini Flash"] = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

print("Benchmarking: Sieve of Eratosthenes code generation\n")
print(f"{'Provider':<20} {'Latency':>10} {'Tokens':>8}")
print("-" * 45)

for name, provider in providers_to_test.items():
    chain = benchmark_prompt | provider
    start = time.time()
    response = chain.invoke(task)
    latency = time.time() - start
    tokens = response.usage_metadata.get('total_tokens', 'N/A')
    print(f"{name:<20} {latency:>9.2f}s {str(tokens):>8}")

---
## ✅ Summary

| Concept | Key Takeaway |
|---|---|
| **ChatModel** | Always use `Chat*` classes — returns `AIMessage`, not string |
| **Provider swapping** | Change the import + model name — chain code unchanged |
| **PromptTemplate** | String templates with `{variables}` — use for sub-components |
| **ChatPromptTemplate** | Use `from_messages([...])` — standard for ChatModels |
| **MessagesPlaceholder** | Inject dynamic conversation history into any prompt |
| **Few-shot** | Embed input→output examples — most reliable format control |
| **Chain-of-Thought** | Ask LLM to reason step-by-step — improves complex reasoning |

**Next**: [03 — Structured Outputs & Parsers](../03_structured_output_and_parsers/theory.md)